# MicrobiomeDataSpace: a data object for functional metagenome 



## An overview


In this tutorial, we will investigate the abundance of nitrate reduction (nar) operon across different disease states in human gut microbiome samples. The nar operon is particularly interesting as it plays a crucial role in anaerobic respiration, allowing bacteria to use nitrate as a terminal electron acceptor. This capability is especially relevant in the context of inflammatory bowel diseases (IBD), where changes in the gut environment can affect microbial metabolism and composition.

The data we will analyze comes from a comprehensive IBD cohort study (https://www.cell.com/cell/pdf/S0092-8674(22)00850-9.pdf), which includes samples from healthy individuals, IBD patients (both Ulcerative Colitis and Crohn's Disease), and other gastrointestinal conditions. By profiling the nar operon abundance, we aim to understand how this important metabolic pathway varies across different disease states and potentially contributes to disease pathogenesis.



In [1]:
import polars as pl
import metabiome.io as mio

## Load data

In [2]:
data_dir = "/biodata/resources/MetaGEAR_vis/nar_operon_query_output"
# Load from multiple file formats
mds = mio.from_files(
    obs="{}/input/metadata".format(data_dir),           # Sample metadata
    abundance="{}/input/RPKM.json".format(data_dir),   # Gene abundance profiles
    taxonomy="{}/input/taxonomy.json".format(data_dir), # Taxonomic annotations
    functional="{}/input/FG.json".format(data_dir),    # Functional groups
    sequences="{}/input/merged.fasta".format(data_dir), # Gene sequences
    id_mapping="{}/input/id_mapping.tsv".format(data_dir) # ID cross-references
)


In [3]:
# Check data dimensions
print(f"Data shape: {mds.shape}")  # (n_samples, n_features)

Data shape: (9522, 9484)


In [4]:
mds.obs.head()  # Display first few rows of sample metadata

sample,disease_group,disease_state,use_antibiotic,age,gender
str,cat,cat,cat,str,cat
"""SRR8865580""","""CRC""",null,"""no""","""67""","""M"""
"""SRR8865577""","""CRC""",null,"""no""","""71""","""M"""
"""SRR8865593""","""CRC""",null,"""no""","""72""","""M"""
"""SRR8865575""","""CRC""",null,"""no""","""60""","""M"""
"""SRR8865594""","""CRC""",null,"""no""","""48""","""M"""


In [5]:
mds.var.head()  # Display first few rows of sample metadata

gc_id,msp_id,gene_category,domain,kingdom,phylum,class,order,family,genus,species,pfam_domain
str,str,str,str,str,str,str,str,str,str,str,cat
"""Israel__10055__k141_1930::8::4…","""NA""","""NA""","""NA""",null,null,null,null,null,null,null,"""PF13247:::PF14711"""
"""cohort_merged__ERR4341682__k14…","""NA""","""NA""","""NA""",null,null,null,null,null,null,null,"""PF02665"""
"""France__9822__k141_49592::4::1…","""NA""","""NA""","""NA""",null,null,null,null,null,null,null,"""PF02613"""
"""cohort_merged__SRR14610645__k1…","""NA""","""NA""","""NA""",null,null,null,null,null,null,null,"""PF02613"""
"""cohort_merged__SRR6367599__k14…","""NA""","""NA""","""NA""",null,null,null,null,null,null,null,"""PF02613"""


## Profile the abundance of the nar operon (per species per sample)

- Step1: group by functional groups and species

- Step2: calculated the total abundance 

- Step3: group by species

- Step4: calculate the median abundance 

In [6]:
# aggregrate by pfam_domain and taxa first
operon_abd_data = mds.groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")

<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> What is the mean abundance of nar operon per species for each disease group? 
</div>

<details>
<summary><strong>🔎 Solution :</strong></summary>

```

operon_abd_data_diseaseMean = operon_abd_data.groupby.obs("disease_group").agg("mean")


```

</details>

In [7]:
# Your solutions here 
operon_abd_data_diseaseMean = operon_abd_data.groupby.obs("disease_group").agg("mean")


## Visualization: barplot to visualize the mean abundance across disease groups

- Select a species of interest

- use operon_abd_data.pl.barplot to plot barplot

In [8]:
cur_species =  "Escherichia coli" 
operon_abd_data.pl.barplot(
    feature_name=cur_species, 
    x_axis_col="disease_group",
    feature_type_col="species",
    title="Abundance of {0} by Disease Group".format(cur_species),
    color_map = {
            "Healthy": "#2166ac",
            "nonIBD": "#2166ac",
            "CD": "#b2182b",
            "UC": "#d6604d",
            "CRC": "#762a83",
            "MP": "#9970ab",
            "adenoma": "#c2a5cf",
        }
)


<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> What is the mean abundance of Veillonella parvula across disease groups
</div>


<details>
<summary><strong>🔎 Solution :</strong></summary>

```

cur_species =  "Veillonella parvula"
operon_abd_data.pl.boxplot(
    feature_name=cur_species, 
    x_axis_col="disease_group",
    feature_type_col="species",
    title="Abundance of {0} by Disease Group".format(cur_species),
    color_map = {
            "Healthy": "#2166ac",
            # "nonIBD": "#2166ac",
            "CD": "#b2182b",
            "UC": "#d6604d",
            # "CRC": "#762a83",
            # "MP": "#9970ab",
            # "adenoma": "#c2a5cf",
        }
)



```

</details>

In [9]:
# Your solutions here
cur_species =  "Veillonella parvula"
operon_abd_data.pl.boxplot(
    feature_name=cur_species, 
    x_axis_col="disease_group",
    feature_type_col="species",
    title="Abundance of {0} by Disease Group".format(cur_species),
    color_map = {
            "Healthy": "#2166ac",
            # "nonIBD": "#2166ac",
            "CD": "#b2182b",
            "UC": "#d6604d",
            # "CRC": "#762a83",
            # "MP": "#9970ab",
            # "adenoma": "#c2a5cf",
        }
)

## Stratify the abundance by disease group and visualize by Sankey plot

In [10]:
# select all healthy samples, the profile the nar operon abundance for each species
operon_abd_data_Healthy = mds.filter.obs(pl.col("disease_group") == "Healthy").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")


/workspaces/env-multiomics-workshop/Metabiome/src/metabiome/core/space.py:369: UserWarning:

1 filtered samples were not present in the matrix and were dropped



In [11]:
# visualize using sankey plot
operon_abd_data_Healthy.pl.sankey(
    hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)


<div style="border-left: 4px solid #007acc; padding: 0.5em; background: rgba(0, 122, 204, 0.1); border-radius: 4px;">
<strong>📝 Question :</strong> Please plot the sankey plot for UC patients, do you observe any changes of the composition?
</div>
 

<details>
<summary><strong>🔎 Solution :</strong></summary>

```

operon_abd_data_UC = mds.filter.obs(pl.col("disease_group") == "UC").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")

operon_abd_data_UC.pl.sankey(
   hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)


```

</details>


In [12]:
# your solutions here
operon_abd_data_UC = mds.filter.obs(pl.col("disease_group") == "UC").groupby.var(["pfam_domain","species"]).agg("sum").groupby.var(["species"]).agg("median")

operon_abd_data_UC.pl.sankey(
   hierarchy_cols=["domain","phylum","class","family","genus", "species"], min_abundance=0.5
)